## 1. Initial Data Inspection

This section performs an initial exploration of the US Regional Sales dataset to understand its structure, size, column characteristics, and overall contents. It examines the dataset dimensions, data types, missing values, unique values, duplicate records, and sample rows to establish a baseline understanding of the data before proceeding with detailed data quality assessment and preparation.

In [1]:
import pandas as pd

path = '../data/raw/US_Regional_Sales_Data.csv'
df = pd.read_csv(path)

def section(title):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")

# 1. Shape
section("SHAPE")
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")
print(f"Exact duplicate rows: {df.duplicated().sum():,}")

# 2. Column summary: dtype, nulls, unique count
section("COLUMN SUMMARY")
summary = pd.DataFrame({
    'Dtype':    df.dtypes.astype(str),
    'Nulls':    df.isna().sum(),
    'Null %':   (df.isna().mean() * 100).round(2),
    'Unique':   df.nunique(),
})
print(summary.to_string())

# 3. Sample rows
section("SAMPLE ROWS (first 3)")
print(df.head(3).T.to_string(header=['Row 0', 'Row 1', 'Row 2']))


SHAPE
Rows:    7,991
Columns: 16
Exact duplicate rows: 0

COLUMN SUMMARY
                    Dtype  Nulls  Null %  Unique
OrderNumber        object      0     0.0    7991
Sales Channel      object      0     0.0       4
WarehouseCode      object      0     0.0       6
ProcuredDate       object      0     0.0      11
OrderDate          object      0     0.0     945
ShipDate           object      0     0.0     966
DeliveryDate       object      0     0.0     966
CurrencyCode       object      0     0.0       1
_SalesTeamID        int64      0     0.0      28
_CustomerID         int64      0     0.0      50
_StoreID            int64      0     0.0     367
_ProductID          int64      0     0.0      47
Order Quantity      int64      0     0.0       8
Discount Applied  float64      0     0.0       7
Unit Cost          object      0     0.0    5252
Unit Price         object      0     0.0     664

SAMPLE ROWS (first 3)
                         Row 0         Row 1         Row 2
OrderNumber

In [2]:
d = df.copy()

date_cols = ['ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate']
money_cols = ['Unit Cost', 'Unit Price']

# Parse dates (day-first) and money (strip thousands separators)
for c in date_cols:
    d[c + '_p'] = pd.to_datetime(d[c], format='%d-%m-%Y', errors='coerce')
for c in money_cols:
    d[c + '_p'] = pd.to_numeric(d[c].astype(str).str.replace(',', '', regex=False), errors='coerce')

def section(title):
    print(f"\n{'=' * 50}\n{title}\n{'=' * 50}")

# 1. Overview
section("DATASET OVERVIEW")
print(f"Rows:                {len(df):,}")
print(f"Unique order numbers: {df['OrderNumber'].nunique():,}")
print(f"Duplicate order nums: {df['OrderNumber'].duplicated().sum():,}")
print(f"Warehouses:           {df['WarehouseCode'].nunique()}")

# 2. Date ranges
section("DATE RANGES & PARSE FAILURES")
dates = pd.DataFrame({
    c: {
        'Min': d[c + '_p'].min().date() if d[c + '_p'].notna().any() else None,
        'Max': d[c + '_p'].max().date() if d[c + '_p'].notna().any() else None,
        'Unparsed': int(d[c + '_p'].isna().sum()),
    } for c in date_cols
}).T
print(dates.to_string())

# 3. Date logic checks
section("DATE LOGIC CHECKS (should all be 0)")
checks = pd.Series({
    'Procured after order':   (d['ProcuredDate_p'] > d['OrderDate_p']).sum(),
    'Order after ship':       (d['OrderDate_p'] > d['ShipDate_p']).sum(),
    'Ship after delivery':    (d['ShipDate_p'] > d['DeliveryDate_p']).sum(),
    'Delivery before order':  (d['DeliveryDate_p'] < d['OrderDate_p']).sum(),
}, name='Violations').astype(int)
print(checks.to_string())

# 4. Money
section("MONEY COLUMNS")
money = pd.DataFrame({
    c: {
        'Min': d[c + '_p'].min(),
        'Max': d[c + '_p'].max(),
        'Unparsed': int(d[c + '_p'].isna().sum()),
    } for c in money_cols
}).T
print(money.round(2).to_string())

# 5. Quantity & discount
section("QUANTITY & DISCOUNT RANGE")
print(f"Order Quantity:   {df['Order Quantity'].min()} to {df['Order Quantity'].max()}")
print(f"Discount Applied: {df['Discount Applied'].min():.2f} to {df['Discount Applied'].max():.2f}")

# 6. Categorical counts
for col, title in [('CurrencyCode', 'CURRENCY'), ('Sales Channel', 'SALES CHANNELS')]:
    section(title)
    vc = df[col].value_counts()
    print(pd.DataFrame({'Count': vc, '%': (vc / vc.sum() * 100).round(1)}).to_string())


DATASET OVERVIEW
Rows:                7,991
Unique order numbers: 7,991
Duplicate order nums: 0
Warehouses:           6

DATE RANGES & PARSE FAILURES
                     Min         Max Unparsed
ProcuredDate  2017-12-31  2020-09-26        0
OrderDate     2018-05-31  2020-12-30        0
ShipDate      2018-06-02  2021-01-24        0
DeliveryDate  2018-06-07  2021-02-02        0

DATE LOGIC CHECKS (should all be 0)
Procured after order     0
Order after ship         0
Ship after delivery      0
Delivery before order    0

MONEY COLUMNS
               Min      Max  Unparsed
Unit Cost    68.68  5498.56       0.0
Unit Price  167.50  6566.00       0.0

QUANTITY & DISCOUNT RANGE
Order Quantity:   1 to 8
Discount Applied: 0.05 to 0.40

CURRENCY
              Count      %
CurrencyCode              
USD            7991  100.0

SALES CHANNELS
               Count     %
Sales Channel             
In-Store        3298  41.3
Online          2425  30.3
Distributor     1375  17.2
Wholesale        893

## 2. Data Quality Assessment

This section investigates the dataset for missing values and examines the data types of individual columns. The objective is to identify potential data quality issues and determine which columns require further validation or transformation before analysis.

In [3]:
# Missing values by column
missing = pd.DataFrame({
    'Missing': df.isna().sum(),
    'Missing %': (df.isna().mean() * 100).round(2),
}).sort_values('Missing', ascending=False)

print(f"\n{'=' * 45}\nMISSING VALUES BY COLUMN\n{'=' * 45}")
print(missing.to_string())

# Final result
total_missing = int(missing['Missing'].sum())
cols_affected = int((missing['Missing'] > 0).sum())
total_cells = df.shape[0] * df.shape[1]

print(f"\n{'=' * 45}\nFINAL RESULT\n{'=' * 45}")
print(f"Total missing cells:  {total_missing:,} of {total_cells:,} ({total_missing / total_cells * 100:.2f}%)")
print(f"Columns affected:     {cols_affected} of {df.shape[1]}")
print(f"Rows with any missing: {int(df.isna().any(axis=1).sum()):,} of {df.shape[0]:,}")

if total_missing == 0:
    print("Verdict: Dataset is complete. No imputation or dropping needed.")
else:
    print(f"Verdict: Most affected column is '{missing.index[0]}' "
          f"({missing['Missing %'].iloc[0]}% missing). Decide whether to impute or drop.")


MISSING VALUES BY COLUMN
                  Missing  Missing %
OrderNumber             0        0.0
Sales Channel           0        0.0
WarehouseCode           0        0.0
ProcuredDate            0        0.0
OrderDate               0        0.0
ShipDate                0        0.0
DeliveryDate            0        0.0
CurrencyCode            0        0.0
_SalesTeamID            0        0.0
_CustomerID             0        0.0
_StoreID                0        0.0
_ProductID              0        0.0
Order Quantity          0        0.0
Discount Applied        0        0.0
Unit Cost               0        0.0
Unit Price              0        0.0

FINAL RESULT
Total missing cells:  0 of 127,856 (0.00%)
Columns affected:     0 of 16
Rows with any missing: 0 of 7,991
Verdict: Dataset is complete. No imputation or dropping needed.


In [4]:
print(df.dtypes)

OrderNumber          object
Sales Channel        object
WarehouseCode        object
ProcuredDate         object
OrderDate            object
ShipDate             object
DeliveryDate         object
CurrencyCode         object
_SalesTeamID          int64
_CustomerID           int64
_StoreID              int64
_ProductID            int64
Order Quantity        int64
Discount Applied    float64
Unit Cost            object
Unit Price           object
dtype: object


## 3. Data Type Conversion and Validation

This section converts date columns into datetime format and monetary columns into numeric format. Conversion results are validated by checking for parsing failures, ensuring that the transformed data is suitable for subsequent analysis while preserving the original dataset.

In [5]:
# Create a working copy; preserve the original DataFrame
clean_df = df.copy()

date_cols = ['ProcuredDate', 'OrderDate', 'ShipDate', 'DeliveryDate']
money_cols = ['Unit Cost', 'Unit Price']

# Convert dates (DD-MM-YYYY)
for col in date_cols:
    clean_df[col] = pd.to_datetime(df[col], format='%d-%m-%Y', errors='coerce')

# Convert money (strip thousands separators)
for col in money_cols:
    clean_df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

# Build the report: compare against the ORIGINAL data
rows = []
for col in date_cols + money_cols:
    originally_missing = int(df[col].isna().sum())
    missing_after = int(clean_df[col].isna().sum())
    rows.append({
        'Column': col,
        'Type': 'date' if col in date_cols else 'money',
        'Dtype before': str(df[col].dtype),
        'Dtype after': str(clean_df[col].dtype),
        'Originally missing': originally_missing,
        'Conversion failures': missing_after - originally_missing,
    })

report = pd.DataFrame(rows).set_index('Column')

print(f"\n{'=' * 75}\nCONVERSION REPORT\n{'=' * 75}")
print(report.to_string())

# Final result
total_failures = int(report['Conversion failures'].sum())
print(f"\n{'=' * 75}\nFINAL RESULT\n{'=' * 75}")
print(f"Columns converted:    {len(report)} ({len(date_cols)} date, {len(money_cols)} money)")
print(f"Conversion failures:  {total_failures:,}")

if total_failures == 0:
    print("Verdict: All values converted cleanly. Safe to proceed.")
else:
    bad = report[report['Conversion failures'] > 0].index.tolist()
    print(f"Verdict: Check these columns before continuing: {bad}")
    # Show examples of values that failed
    for col in bad:
        mask = clean_df[col].isna() & df[col].notna()
        print(f"\nSample failed values in '{col}':")
        print(df.loc[mask, col].head(5).to_string())


CONVERSION REPORT
               Type Dtype before     Dtype after  Originally missing  Conversion failures
Column                                                                                   
ProcuredDate   date       object  datetime64[ns]                   0                    0
OrderDate      date       object  datetime64[ns]                   0                    0
ShipDate       date       object  datetime64[ns]                   0                    0
DeliveryDate   date       object  datetime64[ns]                   0                    0
Unit Cost     money       object         float64                   0                    0
Unit Price    money       object         float64                   0                    0

FINAL RESULT
Columns converted:    6 (4 date, 2 money)
Conversion failures:  0
Verdict: All values converted cleanly. Safe to proceed.
